# Yelp Reviews: Binary Sentiment Analysis with a Pretrained Transformer

Goal: classify Yelp reviews as NEGATIVE or POSITIVE using the default Hugging Face sentiment-analysis pipeline. Evaluate 150 randomly sampled reviews without training a model.

This follows the numbered teaching style of the IMDB reference, but uses pretrained inference instead of building and training a Keras transformer, as required by the assignment.

## 1. Install and import libraries

In [ ]:
%pip install -q transformers torch datasets pandas

In [ ]:
import pandas as pd
from pathlib import Path
from transformers import pipeline

## 2. Load Yelp Reviews dataset

Use the original Yelp binary CSV (label, text) if available. Set `csv_path` to its location. If it is not present, use the public Yelp Polarity test split from Hugging Face. The latter uses labels 0 and 1, so we convert them to the original 1 and 2 convention before applying the assignment mapping.

In [ ]:
csv_path = Path("test.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path, header=None, names=["label", "text"])
    # Also support CSV files with a label,text header.
    if str(df.iloc[0]["label"]).strip().lower() == "label":
        df = df.iloc[1:].copy()
else:
    from datasets import load_dataset
    dataset = load_dataset("fancyzhx/yelp_polarity", split="test")
    df = dataset.to_pandas()
    df["label"] = df["label"] + 1

df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)
print("Dataset shape:", df.shape)
df.head()

## 3. Map labels and randomly sample reviews

- 1 = NEGATIVE
- 2 = POSITIVE

Keep the original text: removing stopwords such as “not” can change the sentiment.

In [ ]:
df = df.dropna(subset=["text"]).copy()
df = df[df["text"].astype(str).str.strip().ne("")]
assert df["label"].isin([1, 2]).all(), "Expected original Yelp labels 1 and 2"
df["true_label"] = df["label"].map({1: "NEGATIVE", 2: "POSITIVE"})

sample = df.sample(n=150, random_state=42).reset_index(drop=True)
print(sample["true_label"].value_counts())
sample.head()

## 4. Load the pretrained Transformer

The pipeline handles tokenization, embeddings, positional information, Transformer encoder layers and sentiment prediction automatically. No model training is needed. CPU inference is sufficient.

In [ ]:
classifier = pipeline("sentiment-analysis", device=-1)
print("Model:", classifier.model.config.name_or_path)

## 5. Make predictions

Use truncation to handle reviews longer than the model supports. Padding for batches is handled internally by the pipeline.

In [ ]:
predictions = classifier(sample["text"].tolist(), truncation=True, batch_size=8)
sample["predicted_label"] = [p["label"].upper() for p in predictions]
sample["confidence"] = [p["score"] for p in predictions]
sample.head()

## 6. Evaluate accuracy

Accuracy = number of correct predictions / total number of reviews.

In [ ]:
correct = (sample["true_label"] == sample["predicted_label"]).sum()
accuracy = correct / len(sample)
print(f"Correct predictions: {correct}/{len(sample)}")
print(f"Accuracy: {accuracy:.2%}")

## 7. Print five example predictions

In [ ]:
for i, row in sample.head(5).iterrows():
    print(f"Review {i + 1}")
    print("Text:", row["text"])
    print("True label:", row["true_label"])
    print("Predicted label:", row["predicted_label"])
    print("Confidence:", round(row["confidence"], 4))
    print("-" * 60)

## 8. Attention vs. Bag-of-Words

Self-attention allows the Transformer to consider relationships between words in context, such as connecting “not” with “good” in “The food was not good.” Unlike Bag-of-Words word counts, the Transformer also uses positional information to distinguish word order and interpret contextual meaning.

## 9. Teaching summary

Raw review → Tokenization → Token and positional embeddings → Transformer self-attention and encoder layers → Sentiment classification → POSITIVE/NEGATIVE.

**Execution note:** Run all cells to obtain measured accuracy and the five example predictions. The dataset and pretrained weights require internet on first use unless already available locally. Long reviews may be truncated, which can affect accuracy.